#### Test RS knn with cosine distance on mean centered data, evaluating with RSME and MAE

In [ ]:
import numpy as np
import pandas as pd
from boardgames_recsys.data.filtering import filter_df
from boardgames_recsys.data.matrix import *
from boardgames_recsys.models.collaborative_filtering import *
import seaborn as sns
from boardgames_recsys.evaluation.ratings import *
from scipy.sparse import csr_matrix

%load_ext autoreload
%autoreload 2

In [ ]:
# import DB et set min_reviews
folder = "database_cleaned"
avis_clean  = pd.read_csv(f"{folder}/avis_clean.csv", index_col=0)
jeux_clean  = pd.read_csv(f"{folder}/jeux_clean.csv", index_col=0)
users       = pd.read_csv(f"trictrac_database/users.csv", names=["Username", "User id"])

min_reviews = 5

In [ ]:
# filter data with the minimum reviews
filtered_data = filter_df(avis_clean, min_reviews)

# we center the ratings for the centered cos version
filtered_centrd_data, avg_df = center_score(filtered_data)

In [ ]:
# get the needed matrixes

# matrix_ratings : sparse matrix for nonzero values only: row = users, cols = games
# mask_ratings : mask for NaN values only -> with matrix_ratings, can get ratings = 0
# users/games_table_assoc : association index - ids for the matrix_use

matrix_ratings, mask_ratings, users_table_assoc, games_table_assoc = get_matrix_user_game(filtered_centrd_data)
matrix_ratings_n, mask_ratings_n, users_table_assoc_n, games_table_assoc_n = get_matrix_user_game(filtered_data)
users_table_assoc

In [ ]:
# we calculate the similarity matrix with cos on the centered values
similarity_matrix = calc_distance_matrix(matrix_ratings, mask_ratings, "cos")
similarity_matrix_n = calc_distance_matrix(matrix_ratings_n, mask_ratings_n, "cos")

# cosine similarity range [-1,1] cosine distance = 1 - similarity, range [0, 2] 

In [ ]:
# for the evaluation, we test on the X most popular users (by number of reviews) 
NB_USERS = 200
users_count = filtered_centrd_data[["User id", "Game id"]].groupby("User id", as_index=True).count().sort_values(by="Game id", ascending=False).head(NB_USERS)

# get the index (matrix) associated to the selected users ids
users = users_table_assoc[users_table_assoc.isin(users_count.index)].index.to_numpy()
type(users)

#### **RMSE** 

In [ ]:
result = np.load("generated_data/rmse_centered_200_cos.npy")

In [ ]:
# calculate rmse DO NOT RUN CELL
vect_rmse = np.vectorize(calc_RMSE_cos, excluded=['matrix_ratings', 'mask_ratings', 'similarity_matrix'])
result = vect_rmse(users, matrix_ratings=matrix_ratings, mask_ratings=mask_ratings, similarity_matrix=similarity_matrix)
np.save(f"generated_data/rmse_centered_{NB_USERS}_cos.npy", result)

In [ ]:
# rsme calculated for each users, RMSE can be above 10
np.unique(result)

In [ ]:
# users_table_assoc[users] = true id of users
# we associate each users with their RMSE and count reviews
rmse_users = pd.DataFrame(zip(users_table_assoc[users], result), columns=["User id", "RMSE"]).merge(users_count, on ="User id")
rmse_users.columns = ["User id", "RMSE", "Count reviews"]
rmse_users.sort_values(by="Count reviews", inplace=True)
rmse_users

In [ ]:
# we plot the RMSE distribution for cos

sns.set_style("darkgrid")
plot = sns.jointplot(data=rmse_users, x="Count reviews", y="RMSE", kind="scatter")
plot.ax_joint.axhline(rmse_users["RMSE"].mean(), color="gray", linestyle="--", label="Mean RMSE")
plot.ax_joint.text(x=50, y=rmse_users["RMSE"].mean(), s=f'{round(rmse_users["RMSE"].mean(), 2)}', color='gray', fontsize=12, ha='right')
plot.figure.suptitle(f'RMSE for {NB_USERS} users, vers. cos centered', fontsize=15)
plot.figure.tight_layout()
plot.ax_joint.legend()

plt.savefig('images/rmse_centered_min5_200_user_cos.png')

In [ ]:
# normalized RMSE 
rmse_users['Normalized RMSE'] = (rmse_users['RMSE'] - rmse_users['RMSE'].min()) / (rmse_users['RMSE'].max() - rmse_users['RMSE'].min())
rmse_users

In [ ]:
# we plot the RMSE normalized distribution

sns.set_style("darkgrid")
plot = sns.jointplot(data=rmse_users, x="Count reviews", y="Normalized RMSE", kind="scatter")
plot.ax_joint.axhline(rmse_users["Normalized RMSE"].mean(), color="gray", linestyle="--", label="Mean RMSE")
plot.ax_joint.text(x=0, y=rmse_users["Normalized RMSE"].mean(), s=f'{round(rmse_users["Normalized RMSE"].mean(), 2)}', color='gray', fontsize=12, ha='center')
plot.figure.suptitle(f'RMSE for {NB_USERS} users, vers. cos centered', fontsize=15)
plot.figure.tight_layout()
plot.ax_joint.legend()
plt.savefig("images/rmse_centrd_min5_200_users_cos.png")

#### **MAE**

In [ ]:
result = np.load("generated_data/mae_centered_200_cos.npy")

In [ ]:
# calculate mae DO NOT RUN CELL
vect_mae = np.vectorize(calc_MAE_cos, excluded=['matrix_ratings', 'mask_ratings', 'similarity_matrix'])
result = vect_mae(users, matrix_ratings=matrix_ratings, mask_ratings=mask_ratings, similarity_matrix=similarity_matrix)
np.save(f"generated_data/mae_centered_{NB_USERS}_cos.npy", result)

In [ ]:
# users_table_assoc[users] = true id of users
# we associate each users with their MAE and count reviews
rmse_users = pd.DataFrame(zip(users_table_assoc[users], result), columns=["User id", "MAE"]).merge(users_count, on ="User id")
rmse_users.columns = ["User id", "MAE", "Count reviews"]
rmse_users.sort_values(by="Count reviews", inplace=True)
# normalized MAE 
rmse_users['Normalized MAE'] = (rmse_users['MAE'] - rmse_users['MAE'].min()) / (rmse_users['MAE'].max() - rmse_users['MAE'].min())
rmse_users

In [ ]:
# we plot the MAE distribution

sns.set_style("darkgrid")
plot = sns.jointplot(data=rmse_users, x="Count reviews", y="MAE", kind="scatter")
plot.ax_joint.axhline(rmse_users["MAE"].mean(), color="gray", linestyle="--", label="Mean MAE")
plot.ax_joint.text(x=0, y=rmse_users["MAE"].mean(), s=f'{round(rmse_users["MAE"].mean(), 2)}', color='gray', fontsize=12, ha='center')
plot.figure.suptitle(f'MAE for {NB_USERS} users, vers. cos centered', fontsize=15)
plot.figure.tight_layout()
plot.ax_joint.legend()

plt.savefig("images/mae_centrd_min5_200_cos")